# 하의(팬츠) 재라벨링 → 재인덱싱 → 카테고리 파인튜닝, 밤새 한 번에

`aihub_bottom_label.py`(신규)로 `04bottom_01pants`(소재 컬럼이 없어 통째로 제외됐던 팬츠)를
재라벨링하고, 그 결과를 `aihub_dataset.py`에 반영해 인덱스를 다시 만든 뒤, 실제 배포 중인
방식과 같은 파인튜닝(`aihub_finetune_vit.py`, last-4 언프리즈)까지 한 번에 돌린다.

**이 노트북이 재현하는 파이프라인은 `aihub_pipeline_server.ipynb`에서 실제로 돌았던 명령을
그대로 가져온 것이다** — 하이퍼파라미터를 새로 추측하지 않았다(`--unfreeze-layers 4
--epochs 10 --batch-size 32`, `--variant masked`, `--lr-backbone`은 스크립트 기본값 1e-5로
그 노트북과 동일). 다른 점은 STEP 2(인덱싱) 전에 하의 재라벨링 3단계가 끼어든다는 것뿐이다.

## 쓰는 법

1. 아래 "0. 경로 설정" 셀에서 `CLOTHES_ROOT`/`ACCESSORY_ROOT`/`IMAGE_ROOT`를 서버의 실제
   경로로 고친다.
2. **먼저 `SMOKE_LIMIT = 50`으로 한 번 "Run All"** — 라벨링→재인덱싱→캐싱→파인튜닝 전
   구간이 몇 분 안에 끝까지 도는지, 에러 없이 배선이 맞는지 확인한다. 이 저장소엔 GPU도
   AI-Hub 데이터도 없어서 이 노트북 자체를 실행해 보고 만든 게 아니다 — 반드시 이 스모크
   런으로 먼저 검증할 것.
3. 스모크 런이 깨끗하면 `SMOKE_LIMIT = None`으로 바꾸고 다시 "Run All" → 그대로 자면 된다.
4. STEP 8(파인튜닝)이 가장 오래 걸린다(GPU, epochs=10). 나머지는 전부 그보다 훨씬 짧다.

## 의존성

바로 아래 셀이 설치한다(`requirements-test.txt` + `bitsandbytes`) — 셸에서 따로 설치할 필요
없다. fashion_clip·Qwen3-VL-4B-Instruct 둘 다 **공개 체크포인트**라 HF 토큰도 필요 없다
(SAM3와 다르게 gated 아님).

## 밤사이 뭘 만드는지

- `artifacts/bottom_pass1.jsonl` / `bottom_pass2.jsonl` / `bottom_merged.jsonl` — 재라벨링 결과
- `artifacts/index.jsonl` / `sample.jsonl` — 하의 포함해서 다시 만든 인덱스
- `artifacts/pixel_cache_*` — 파인튜닝용 입력 텐서 캐시
- `artifacts/head_category_masked.joblib` — 빠른 선형 헤드 베이스라인(참고용)
- `artifacts/artifacts_finetune/vit_last4_category_masked_lr1e-05/` — **오늘 밤의 진짜 결과물**
  (`checkpoint.pt` + `meta.json`, epoch별 val accuracy 이력 포함)

## 낮에 할 일 (이 노트북은 안 함)

- `aihub_evaluate.py --ground-truth-dir product-info-check`로 정답셋 정식 평가(이미지가
  gitignore라 서버에 따로 올려야 함) — 사람 판단이 필요해 자동화 대상에서 뺐다.
- 결과가 괜찮으면 체크포인트를 `worker/models/product_classifier/`로 반영할지 결정.


In [ ]:
# 의존성 설치 — 이 노트북 안에서 전부 해결한다(셸에서 따로 설치 안 해도 됨).
#
# `validation/requirements-test.txt`를 그대로 쓰지 않는다 — 그 파일은 transformers<5로
# 고정돼 있는데(다른 스파이크의 Kanana 모델 호환 때문), Qwen3-VL(`aihub_bottom_label.py`의
# qwen3_vl 백엔드)은 transformers>=5.0이 있어야 `Qwen3VLForConditionalGeneration` 클래스가
# 존재한다. 이 파이프라인엔 그 파일의 peft/open_clip/optimum/datasets 등도 필요 없어서
# (fashion_clip은 순정 transformers AutoModel로 충분, LoRA/marqo/ONNX 경로는 안 씀),
# 실제로 쓰는 패키지만 최소로 설치한다.
%pip install -q "torch>=2.4" "transformers>=5.0" "accelerate>=0.34" "bitsandbytes>=0.50" "scikit-learn>=1.5" "Pillow>=10.0" numpy joblib
print("설치 완료")


## 0. 경로 설정 + 실행 헬퍼

`aihub_pipeline_server.ipynb`와 같은 패턴 — 상대경로/`Path.cwd()`를 안 쓰는 이유도 동일(원격 커널은 파일은 로컬·실행은 원격이라 상대경로가 어긋난다, 브리핑 §3-1).

In [ ]:
import json
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path

# ── 여기만 서버 환경에 맞게 고친다 ────────────────────────────────────
HERE = Path("/home/j-i15a108/work/S15P11A108/AI/validation/CLASS/product-info-finetune")
assert (HERE / "aihub_dataset.py").exists(), f"HERE가 틀렸다: {HERE}"

CLOTHES_ROOT = Path("/home/j-i15a108/work/aihub/extracted/clothes")
ACCESSORY_ROOT = Path("/home/j-i15a108/work/aihub/extracted/accessory_fixed")
IMAGE_ROOT = Path("/home/j-i15a108/work/aihub/extracted/images")
ARTIFACTS = HERE / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

VARIANT = "masked"
SHORT_PANTS_MAX_CM = None  # 기존 배포 파이프라인(aihub_pipeline_server.ipynb)과 동일하게 미설정

# --- 스모크 테스트 스위치 --- 반드시 먼저 작은 값으로 한 번 전체를 돌려볼 것(위 안내 참고).
SMOKE_LIMIT = 50  # None으로 바꾸면 전체 데이터로 진짜 밤샘 실행

# 2차(qwen3_vl) 패스 모델. 이 서버 GPU(NVIDIA L40S, 47.7GB VRAM) 기준으로 골랐다 —
# 실측 벤치마크상 8B(114 tok/s)와 30B-A3B(110.8 tok/s, MoE라 토큰당 활성 파라미터는
# ~3B뿐)의 생성 속도가 거의 같아 속도 손해 없이 표현력만 훨씬 큰 모델을 쓸 수 있다.
# 다만 VRAM은 총 가중치(30B) 기준으로 든다 — bf16 풀로드(~60GB)는 47.7GB에
# 안 들어간다. 공식 FP8 체크포인트("-FP8" 접미사)는 transformers가 아직 직접
# 로딩을 지원하지 않는다(2026-08 확인, vLLM/SGLang 전용) — 쓰지 말 것.
# 대신 bitsandbytes 8bit(추가 설치 불필요)로 로드 시점 양자화한다 — ~30GB대만 써서
# 47.7GB에 여유 있게 들어가면서도 4bit보다 정밀도 손실이 훨씬 적다.
#
# **스모크 테스트(SMOKE_LIMIT=50)에서 OOM이 나면 즉시 아래로 낮출 것**
#   1순위 폴백: --quantization "4bit" (같은 30B-A3B, bitsandbytes nf4, ~16GB급)
#   2순위 폴백: "Qwen/Qwen3-VL-8B-Instruct" (dense, 항상 안전, 8B급 VRAM)
#   3순위 폴백: "Qwen/Qwen3-VL-4B-Instruct" (aihub_bottom_label.py 기본값, 8GB급 GPU도 OK)
QWEN3_VL_MODEL_ID = "Qwen/Qwen3-VL-30B-A3B-Instruct"
QWEN3_VL_QUANTIZATION = "8bit"

print(f"HERE={HERE}")
print(f"SMOKE_LIMIT={SMOKE_LIMIT} ({'스모크 테스트 모드' if SMOKE_LIMIT else '전체 실행 모드'})")


def run(script, *args):
    """스크립트를 실행하고 출력을 그대로 보여준다(aihub_pipeline_server.ipynb와 동일 패턴)."""
    cmd = [sys.executable, str(HERE / script), *[str(a) for a in args]]
    print("$ " + " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8")
    print(result.stdout)
    if result.returncode != 0:
        print("--- stderr ---")
        print(result.stderr)
    return result


def run_or_stop(script, *args):
    """실패하면 그 자리에서 예외를 던진다 — 뒤 단계가 앞 단계 산출물에 의존하는
    구간(재라벨링→병합→재인덱싱→파인튜닝)에 쓴다. 실패한 걸 모르고 밤새 엉뚱한 걸
    돌리는 것보다, 이 시점에서 멈춰서 아침에 바로 원인을 보는 게 낫다."""
    result = run(script, *args)
    if result.returncode != 0:
        raise RuntimeError(f"{script} 실패(code={result.returncode}) — 위 stderr 확인")
    return result


## STEP 1 — 1차 인덱스 (하의 라벨 없이)

`aihub_bottom_label.py`가 "하의인데 subcategory가 없는 행"을 찾으려면 먼저 `index.jsonl`이
있어야 한다. 이미 최근 `index.jsonl`이 있으면(다른 작업으로) 이 셀은 건너뛰어도 되지만,
`--category-cap`/`--accessory-cap` 등이 최신인지 보장하려고 기본은 다시 만든다.

In [ ]:
REBUILD_INITIAL_INDEX = True  # 이미 최신 index.jsonl이 있으면 False로 건너뛴다

if REBUILD_INITIAL_INDEX:
    index_args = [
        "index",
        "--root", CLOTHES_ROOT,
        "--accessory-root", ACCESSORY_ROOT,
        "--out-dir", ARTIFACTS,
        "--category-cap", 3000,
        "--accessory-cap", 3000,
        "--color-cap", 2000,
        "--eval-cap", 100,
        "--accessory-shot-type", "product",
    ]
    if SHORT_PANTS_MAX_CM is not None:
        index_args += ["--short-pants-max-cm", SHORT_PANTS_MAX_CM]
    # SMOKE_LIMIT을 여기 적용하지 않는다 — aihub_dataset.py의 iter_label_files()는
    # 디렉터리를 정렬된 순서로 순회하고 --limit이 그 순서 맨 앞에서 자른다. 04bottom_01pants는
    # 01outer_*/02top_* 등보다 알파벳순으로 뒤라서, 작은 --limit이면 하의가 한 건도 안 잡혀
    # STEP 2~3(하의 재라벨링) 자체를 스모크 테스트할 수 없다(실측: --limit 50 -> 하의 0건).
    # 원본 JSON 스캔은 순수 I/O라 전체를 돌려도 몇 분 안에 끝난다 -- 느린 건 VLM
    # 재라벨링(STEP 3)·파인튜닝(STEP 8)이고, 그쪽은 이미 별도로 SMOKE_LIMIT이 걸려 있다.
    run_or_stop("aihub_dataset.py", *index_args)
else:
    assert (ARTIFACTS / "index.jsonl").exists(), "index.jsonl이 없다 — REBUILD_INITIAL_INDEX=True로 먼저 만들 것"
    print("건너뜀 — 기존 index.jsonl 사용")


## STEP 2 — 하의 1차 재라벨링 (fashion_clip, 빠름)

`04bottom_01pants`이면서 아직 `subcategory`가 없는 행 전체를 대상으로 한다. GPU에서
수천 장/분이라 대부분의 시간은 여기가 아니라 STEP 8(파인튜닝)에서 쓴다.

In [ ]:
bottom_pass1_path = ARTIFACTS / "bottom_pass1.jsonl"

pass1_args = [
    "--index", ARTIFACTS / "index.jsonl",
    "--image-root", IMAGE_ROOT,
    "--out", bottom_pass1_path,
    "--backend", "fashion_clip",
]
if SMOKE_LIMIT:
    pass1_args += ["--limit", SMOKE_LIMIT]

run_or_stop("aihub_bottom_label.py", *pass1_args)


## STEP 3 — 애매한 것만 2차 재라벨링 (qwen3_vl)

1차에서 `--min-confidence`(기본 0.5) 미만이라 `accepted=false`였던 행만 골라 생성형 VLM으로
다시 물어본다. 장당 수 초라 전체가 아니라 이 서브셋에만 쓴다.

GPU가 여유 있으면 위 0번 셀의 `QWEN3_VL_MODEL_ID`/`QWEN3_VL_QUANTIZATION`을 바꿔 더
큰 체크포인트로 정확도를 올릴 수 있다(속도는 그만큼 더 느려진다 — 2차 패스는 이미
애매한 소수만 대상이라 감내할 만하다).

In [ ]:
bottom_pass2_path = ARTIFACTS / "bottom_pass2.jsonl"

pass2_args = [
    "--index", ARTIFACTS / "index.jsonl",
    "--image-root", IMAGE_ROOT,
    "--out", bottom_pass2_path,
    "--backend", "qwen3_vl",
    "--model-id", QWEN3_VL_MODEL_ID,
    "--quantization", QWEN3_VL_QUANTIZATION,
    "--only-uncertain", bottom_pass1_path,
]
if SMOKE_LIMIT:
    pass2_args += ["--limit", SMOKE_LIMIT]

run_or_stop("aihub_bottom_label.py", *pass2_args)


## STEP 4 — 병합 + 재인덱싱 (하의 포함)

1차·2차 출력을 이어붙인다 — 같은 `label_path`가 양쪽에 있으면 **2차(qwen3_vl) 결과가
이긴다**(`load_bottom_pseudo_labels()`는 dict라 나중에 읽은 값이 덮어쓴다, 2차 파일을 뒤에
이어붙였으니 자동으로 그렇게 된다). 그 다음 `aihub_dataset.py index`를 `--bottom-pseudo-labels`
와 함께 다시 돌려 `index.jsonl`/`sample.jsonl`에 하의 소분류를 반영한다.

In [ ]:
bottom_merged_path = ARTIFACTS / "bottom_merged.jsonl"

with bottom_merged_path.open("w", encoding="utf-8") as out:
    for part in (bottom_pass1_path, bottom_pass2_path):
        if part.exists():
            out.write(part.read_text(encoding="utf-8"))

reindex_args = [
    "index",
    "--root", CLOTHES_ROOT,
    "--accessory-root", ACCESSORY_ROOT,
    "--out-dir", ARTIFACTS,
    "--category-cap", 3000,
    "--accessory-cap", 3000,
    "--color-cap", 2000,
    "--eval-cap", 100,
    "--accessory-shot-type", "product",
    "--bottom-pseudo-labels", bottom_merged_path,
]
if SHORT_PANTS_MAX_CM is not None:
    reindex_args += ["--short-pants-max-cm", SHORT_PANTS_MAX_CM]
# 위 STEP 1과 같은 이유로 여기도 SMOKE_LIMIT을 안 건다 — 재인덱싱은 하의 포함 전체
# 코퍼스를 다시 스캔해야 STEP 5 정합성 확인이 의미가 있다.

run_or_stop("aihub_dataset.py", *reindex_args)


## STEP 5 — 정합성 확인 (하의가 실제로 들어왔는지)

여기서 5종 전부 0건이면 STEP 6~8(캐싱·파인튜닝)을 몇 시간 돌려도 오늘 밤의 목표(하의 분류
보완)는 달성되지 않는다 — 그대로 진행은 하되, 아침에 바로 알아챌 수 있게 크게 찍는다.

In [ ]:
PANTS_MATERIAL_CODES = ("DENIM", "SLACKS", "CARGO_PANTS", "JOGGER_PANTS", "COTTON_PANTS")

sample_rows = []
with (ARTIFACTS / "sample.jsonl").open(encoding="utf-8") as f:
    sample_rows = [json.loads(line) for line in f if line.strip()]

counts = Counter(r["subcategory"] for r in sample_rows if r.get("subcategory") in PANTS_MATERIAL_CODES)
print("층화 샘플(sample.jsonl)의 하의 소재 분포:")
for code_ in PANTS_MATERIAL_CODES:
    print(f"  {code_:15s} {counts.get(code_, 0):,}")

if sum(counts.values()) == 0:
    print()
    print("!" * 60)
    print("!!! 경고: 하의 소재 코드가 sample.jsonl에 하나도 없다.")
    print("!!! bottom_merged.jsonl 내용, --index 경로, label_path 일치 여부를 확인할 것.")
    print("!" * 60)
    print()


## STEP 6 — 빠른 선형 헤드 베이스라인 (참고용)

`aihub_finetune_vit.py`(STEP 8, 실제 배포 방식)는 오래 걸린다. 그 전에 훨씬 빠른 순수
선형 프로브(동결 임베딩 + sklearn 로지스틱 회귀)로 "하의 데이터가 최소한 신호가 있는지"를
먼저 본다 — 여기서 카테고리 정확도가 말이 안 되게 낮으면(예: 하의만 0에 가까움) STEP 8을
몇 시간 태우기 전에 STEP 2~5를 다시 볼 근거가 된다. **이 헤드 자체가 최종 산출물은 아니다**
— `worker/models/product_classifier/`에 실제로 얹히는 건 STEP 8 쪽(CH027 확인 사항).

In [ ]:
embed_start = time.time()
for split in ("train", "val", "test"):
    embed_args = [
        "--index", ARTIFACTS / "sample.jsonl",
        "--label-root", CLOTHES_ROOT,
        "--accessory-label-root", ACCESSORY_ROOT,
        "--image-root", IMAGE_ROOT,
        "--out-dir", ARTIFACTS,
        "--split", split,
        "--variant", VARIANT,
        "--batch-size", 64,
        "--overwrite",
    ]
    if SMOKE_LIMIT:
        embed_args += ["--limit", SMOKE_LIMIT]
    run("aihub_embed.py", *embed_args)  # 실패해도 STEP 8 진행에는 지장 없어 run_or_stop 안 씀

print(f"임베딩 추출 소요: {(time.time() - embed_start) / 60:.1f}분")

run(
    "aihub_train_head.py",
    "--artifacts", ARTIFACTS,
    "--task", "category",
    "--variant", VARIANT,
    "--class-weight", "balanced",
)


## STEP 7 — 입력 텐서 캐싱 (파인튜닝용, GPU 불필요)

`aihub_finetune_vit.py`가 읽을 `pixel_values` 텐서를 미리 만든다. 모델을 안 실어서(전처리만)
CPU로도 돈다.

In [ ]:
cache_start = time.time()
for split in ("train", "val", "test"):
    cache_args = [
        "--index", ARTIFACTS / "sample.jsonl",
        "--label-root", CLOTHES_ROOT,
        "--accessory-label-root", ACCESSORY_ROOT,
        "--image-root", IMAGE_ROOT,
        "--out-dir", ARTIFACTS,
        "--split", split,
        "--variant", VARIANT,
    ]
    run_or_stop("aihub_cache_pixels.py", *cache_args)

print(f"픽셀 캐싱 소요: {(time.time() - cache_start) / 60:.1f}분")


## STEP 8 — 카테고리 헤드 파인튜닝 (last-4 언프리즈, GPU) — **오늘 밤의 진짜 목표**

`aihub_pipeline_server.ipynb`에서 실제로 배포된 체크포인트를 만든 것과 **완전히 같은 명령**
(`--unfreeze-layers 4 --epochs 10 --batch-size 32`, `--lr-backbone`은 스크립트 기본값
1e-5). 스모크 테스트가 아니라면 여기가 가장 오래 걸리는 단계다 — GPU·데이터 규모에 따라
수십 분~수 시간 예상(정확한 시간은 실측해 봐야 안다, 추측 근거 없음).

In [ ]:
finetune_start = time.time()
finetune_args = [
    "--pixel-cache-dir", ARTIFACTS,
    "--task", "category",
    "--variant", VARIANT,
    "--unfreeze-layers", 4,
    "--epochs", 2 if SMOKE_LIMIT else 10,  # 스모크 모드에서는 2 epoch만 — 배선 확인이 목적
    "--batch-size", 32,
]
run_or_stop("aihub_finetune_vit.py", *finetune_args)
print()
print(f"파인튜닝 소요: {(time.time() - finetune_start) / 60:.1f}분")


## 산출물 확인 + epoch별 학습 곡선

In [ ]:
print("=== artifacts/ 파일 목록 ===")
for path in sorted(ARTIFACTS.glob("*")):
    size = path.stat().st_size if path.is_file() else 0
    print(f"{size / 1e6:10.2f} MB  {path.name}")

finetune_dir = ARTIFACTS / "artifacts_finetune" / f"vit_last4_category_{VARIANT}_lr1e-05"
meta_path = finetune_dir / "meta.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    print()
    print(f"=== {meta_path} ===")
    print(json.dumps(meta, ensure_ascii=False, indent=2))
else:
    print()
    print(f"(meta.json 없음 — 예상 경로: {meta_path}, 실제 폴더명은 하이퍼파라미터로 자동 생성되니 "
          f"위 파일 목록에서 'vit_last4_category'로 시작하는 폴더를 직접 확인할 것)")


## 낮에 할 일

1. 위 `meta.json`의 epoch별 val accuracy와 test accuracy를 확인한다. 특히 하의 5종
   (DENIM/SLACKS/CARGO_PANTS/JOGGER_PANTS/COTTON_PANTS)의 클래스별 정확도 — 전체 평균만
   보면 하의가 여전히 나쁜데 다른 클래스에 묻혀 안 보일 수 있다.
2. `aihub_evaluate.py --ground-truth-dir product-info-check`로 우리 정답셋 기준 정식 평가
   (정답셋 이미지가 서버에 있어야 한다 — gitignore라 따로 올렸어야 함).
3. STEP 6 선형 프로브 결과와 STEP 8 파인튜닝 결과를 나란히 놓고, 하의 쪽에서 파인튜닝이
   실제로 더 나은지 확인한다(더 안 좋으면 `--unfreeze-layers`/`--epochs`를 의심).
4. 결과가 기준(브리핑 §11 수용 기준 참고)을 넘으면 체크포인트를
   `worker/models/product_classifier/`로 반영할지 판단 — CH027이 지적했듯 그 반영 게이트가
   지금 기록에 안 남아 있으니, 이번엔 통과 여부를 `REPORT.md`나 이 노트북 결과 셀에 남겨 둘 것.
